
# AGN composite SED: per-block decomposition

A single ``log L_bol = 12.5`` composable AGN built up component by
component — disc alone, +torus, +narrow lines, +broad lines — so the
reader can see what each block contributes to the total spectrum.
The bottom panel shows the same decomposition stacked so the layers
add up to the full SED.

This is the diagnostic figure for "where does the AGN signal in my
data come from?" — broad-line decompositions need ``blr``, NLR
fitters need ``nlr``, NIR/MIR color fitters need ``torus`` (and
disc choice barely matters longward of 1 μm), etc.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18

ssp = tengri.load_ssp()
COMMON = dict(
    sfh={"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0},
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    redshift=tengri.Fixed(0.05),
)
BASE_AGN = dict(disc={"type": "multicolor", "all_params": tengri.FIXED})


def _agn(extra_blocks=(), wave=None):
    agn = {"all_params": tengri.FIXED, "log_lbol": 12.5, "lum_ratio": 1.0, **BASE_AGN}
    for key, value in extra_blocks:
        agn[key] = value
    model = tengri.SEDModel.build(ssp, agn=agn, **COMMON)
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    # Pin every config to a shared rest grid so the panels can subtract and
    # overplot SEDs: components such as the GRAHSP FeII template otherwise
    # inject their own wavelength nodes, giving each config a different-length
    # native grid (5994 vs 6127).
    lnu = np.asarray(model.predict(p).rest_sed())
    grid = np.asarray(model.wavelengths)
    if wave is None:
        return grid, lnu
    return np.asarray(wave), np.interp(np.asarray(wave), grid, lnu)


configs = [
    ("disc only", (), "#8b4513"),
    (
        "+ torus (SKIRTOR)",
        (("torus", {"type": "skirtor", "all_params": tengri.FIXED}),),
        "#cc7733",
    ),
    (
        "+ NLR",
        (
            ("torus", {"type": "skirtor", "all_params": tengri.FIXED}),
            ("nlr", {"type": "analytic", "all_params": tengri.FIXED}),
        ),
        "#5588cc",
    ),
    (
        "+ BLR",
        (
            ("torus", {"type": "skirtor", "all_params": tengri.FIXED}),
            ("blr", {"type": "analytic", "all_params": tengri.FIXED}),
        ),
        "#4477aa",
    ),
    (
        "+ FeII",
        (
            ("torus", {"type": "skirtor", "all_params": tengri.FIXED}),
            ("blr", {"type": "analytic", "all_params": tengri.FIXED}),
            ("feii", {"type": "grahsp", "all_params": tengri.FIXED}),
        ),
        "#dd6699",
    ),
]

# Establish the shared rest grid from the richest config (all blocks active,
# so it spans every component's wavelength coverage), then evaluate every
# config on it.
common_wave, _ = _agn(configs[-1][1])

seds = {}
for label, blocks, _ in configs:
    _, s = _agn(blocks, wave=common_wave)
    seds[label] = s

wave = common_wave
nu = C_AA_PER_S / wave

fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    figsize=(7.4, 6.8),
    sharex=True,
    gridspec_kw={"hspace": 0.05, "height_ratios": [3, 2]},
)

for label, _, color in configs:
    ax_top.loglog(wave, nu * seds[label], color=color, lw=1.5, label=label)
ax_top.set(ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]", ylim=(1e42, 5e46))
ax_top.legend(frameon=False, fontsize=8, loc="lower center")

disc_only = seds["disc only"]
torus_contrib = seds["+ torus (SKIRTOR)"] - disc_only
nlr_contrib = seds["+ NLR"] - seds["+ torus (SKIRTOR)"]
blr_contrib = seds["+ BLR"] - seds["+ torus (SKIRTOR)"]
feii_contrib = seds["+ FeII"] - seds["+ BLR"]

ax_bot.loglog(wave, nu * disc_only, color="#8b4513", lw=1.2, label="disc")
ax_bot.loglog(
    wave,
    nu * np.where(torus_contrib > 0, torus_contrib, np.nan),
    color="#cc7733",
    lw=1.2,
    label="torus",
)
ax_bot.loglog(
    wave,
    nu * np.where(nlr_contrib > 0, nlr_contrib, np.nan),
    color="#5588cc",
    lw=1.2,
    label="NLR lines",
)
ax_bot.loglog(
    wave,
    nu * np.where(blr_contrib > 0, blr_contrib, np.nan),
    color="#4477aa",
    lw=1.2,
    ls=":",
    label="BLR lines",
)
ax_bot.loglog(
    wave,
    nu * np.where(feii_contrib > 0, feii_contrib, np.nan),
    color="#dd6699",
    lw=1.2,
    label="FeII",
)
ax_bot.set(
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu^{\rm block}$  [erg s$^{-1}$]",
    xlim=(80, 2e6),
    ylim=(1e42, 5e46),
)
ax_bot.legend(frameon=False, fontsize=8, loc="lower right")

plt.savefig("plot_agn_components_breakdown.png", dpi=150, bbox_inches="tight")